In [ ]:
# Week 2 PRoject
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1. Generate Synthetic Highly Imbalanced Financial Data
# ---------------------------------------------------------
X, y = make_classification(
    n_samples=50000,
    n_features=10,
    n_clusters_per_class=1,
    weights=[0.9983, 0.0017],  # Real-world fraud ratio (99.83% vs 0.17%)
    random_state=42,
)

In [ ]:
# Stratified split ensures the exact 99.83/0.17 ratio is kept in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training Class Distribution: {np.bincount(y_train)}")
print(f"Testing Class Distribution: {np.bincount(y_test)}\n")

Training Class Distribution: [39744   256]
Testing Class Distribution: [9936   64]



In [ ]:
# 2. Pipeline 1: Logistic Regression (Requires Scaling)
# ---------------------------------------------------------
print("=" * 60)
print("TRAINING PIPELINE 1: LOGISTIC REGRESSION")
print("=" * 60)

lr_pipeline = ImbPipeline(
    [
        ("scaler", StandardScaler()),  # LR is sensitive to feature scales
        ("smote", SMOTE(random_state=42)),  # Balanced inside training folds
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

TRAINING PIPELINE 1: LOGISTIC REGRESSION


In [ ]:
# Holistic Hyperparameter Tuning via GridSearchCV
lr_param_grid = {
    "smote__k_neighbors": [3, 5],
    "classifier__C": [0.01, 0.1, 1.0],
}

lr_grid = GridSearchCV(
    lr_pipeline, lr_param_grid, cv=5, scoring="roc_auc", n_jobs=-1
)
lr_grid.fit(X_train, y_train)

print(f"Best LR Parameters: {lr_grid.best_params_}")

Best LR Parameters: {'classifier__C': 0.01, 'smote__k_neighbors': 3}


In [ ]:
# 3. Pipeline 2: Random Forest (No Scaling Required)
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("TRAINING PIPELINE 2: RANDOM FOREST")
print("=" * 60)


TRAINING PIPELINE 2: RANDOM FOREST


In [ ]:
rf_pipeline = ImbPipeline(
    [
        (
            "smote",
            SMOTE(random_state=42),
        ),  # No scaler needed for tree-based models
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

In [ ]:
rf_param_grid = {
    "smote__k_neighbors": [5],
    "classifier__max_depth": [10, 20, None],
}

In [ ]:
rf_param_grid = {
    "smote__k_neighbors": [5],

    "classifier__max_depth": [10, 15],

    "classifier__n_estimators": [50],
}

In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# 1. Pipeline define karein
rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 2. Parameters ko Colab ke liye short karein
rf_param_grid = {
    'smote__k_neighbors': [5],
    'classifier__max_depth': [10, 15],
    'classifier__n_estimators': [50]
}

# 3. GridSearchCV mein cv=3 aur verbose=2 lagayein
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

# Run karein
rf_grid.fit(X_train, y_train)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [10, 15],
                         'classifier__n_estimators': [50],
                         'smote__k_neighbors': [5]},
             scoring='roc_auc', verbose=2)

In [ ]:
print(f"Best RF Parameters: {rf_grid.best_params_}\n")

Best RF Parameters: {'classifier__max_depth': 15, 'classifier__n_estimators': 50, 'smote__k_neighbors': 5}



In [ ]:
# 4. Strict Evaluation (Discarding Misleading Accuracy)
# ---------------------------------------------------------
def evaluate_model(model, X_test, y_test, name):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    print("-" * 50)
    print(f" EVALUATION REPORT: {name} ")
    print("-" * 50)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))

    print("\nClassification Metrics:")
    # Discarding global accuracy, prioritizing strict Precision and Recall
    print(classification_report(y_test, preds, target_names=["Legit", "Fraud"]))

    roc_auc = roc_auc_score(y_test, probs)
    print(f"ROC-AUC Score: {roc_auc:.4f}\n")


# Evaluate both architectures on completely untouched test data
evaluate_model(lr_grid, X_test, y_test, "Logistic Regression (with SMOTE)")
evaluate_model(rf_grid, X_test, y_test, "Random Forest (with SMOTE)")

--------------------------------------------------
 EVALUATION REPORT: Logistic Regression (with SMOTE) 
--------------------------------------------------

Confusion Matrix:
[[7845 2091]
 [  30   34]]

Classification Metrics:
              precision    recall  f1-score   support

       Legit       1.00      0.79      0.88      9936
       Fraud       0.02      0.53      0.03        64

    accuracy                           0.79     10000
   macro avg       0.51      0.66      0.46     10000
weighted avg       0.99      0.79      0.88     10000

ROC-AUC Score: 0.7204

--------------------------------------------------
 EVALUATION REPORT: Random Forest (with SMOTE) 
--------------------------------------------------

Confusion Matrix:
[[8521 1415]
 [  33   31]]

Classification Metrics:
              precision    recall  f1-score   support

       Legit       1.00      0.86      0.92      9936
       Fraud       0.02      0.48      0.04        64

    accuracy                          